# Dataset building — customer panels (week / month / year)

Builds clean **customer-period panels** (one row per customer per time period) from the raw
`.Rdata` files. The period granularity is configurable — `week`, `month`, or `year` — and the
layout matches `Datasets/Dataset_clean/electronics_customer_week_panel.csv`:

| column | meaning |
|---|---|
| `Id` | customer id |
| `year` (+ `week` or `month`) | time grid; `week` = `dayofyear // 7` (clipped to 51), `month` = `1..12`, `year`-level has no sub-column |
| `Transactions` | per-period transaction count (the model target) |
| *covariates* | only those that actually exist in the source |

It builds one file per (dataset, frequency), e.g. `apparel_customer_week_panel.csv`.

**Why R is used to read the files:** the `.Rdata` are RDX3 files written by `data.table`;
`pyreadr` (the reader used elsewhere) fails on several of them. We use R as a *thin extractor*
only (dump the needed columns to CSV); all panel-building logic stays in pandas.

**What counts as a transaction:** for most datasets each `mydata` row is already one purchase,
so we count rows. **Multichannel** is line-item level (several rows per order), so there we
count **distinct orders** (`ORDER_NO`) instead — configured per dataset via `tx_id`.

**Covariate note:** only `electronics` and `apparel` carry real covariates (`Gender`/`Income`
exist only in electronics; `high.season` in electronics + apparel). We do **not** fabricate
missing covariates. `electronicV2` is rebuilt by this pipeline as a **validation** panel and
diffed against the known-good `electronics_customer_week_panel.csv` at the end.


In [1]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repo root by walking up until we find the Datasets/ folder, so the
# notebook runs whether launched from the repo root or from its own subfolder.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "Datasets").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "Datasets").exists(), "Could not locate the Datasets/ folder."

DATASETS_DIR = REPO_ROOT / "Datasets"
OUT_DIR = DATASETS_DIR / "Dataset_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Per dataset: output name, source .Rdata, covariates it actually has, and the transaction
# unit. tx_id=None counts mydata rows (one row = one purchase); tx_id="ORDER_NO" counts
# distinct orders per period (Multichannel is line-item level, so rows != transactions).
DATASETS = [
    {"name": "apparel",            "rdata": "Apparel Retailer data.Rdata",        "covariates": ("high.season",),                 "tx_id": None},
    {"name": "book_store",         "rdata": "Book Store data.Rdata",             "covariates": (),                                "tx_id": None},
    {"name": "electronicV2",       "rdata": "Electronics_Retailer_data.Rdata",   "covariates": ("Gender", "Income", "high.season"), "tx_id": None},
    {"name": "sporting_equipment", "rdata": "Sporting Equipment.Rdata",          "covariates": (),                                "tx_id": None},
    {"name": "gift_retailer",      "rdata": "Gift Retailer data (weekly).Rdata", "covariates": (),                                "tx_id": None},
    {"name": "multichannel",       "rdata": "Multichannel Retailer data.Rdata",  "covariates": (),                                "tx_id": "ORDER_NO"},
]

# Frequencies to build for every dataset.
FREQS = ("week", "month", "year")

print("Repo root :", REPO_ROOT)
print("Output dir:", OUT_DIR)


Repo root : /home/virthian/Desktop/Thesis/Package_Notebook
Output dir: /home/virthian/Desktop/Thesis/Package_Notebook/Datasets/Dataset_clean


## Step 1 — Extract raw tables from `.Rdata` (via R)

`extract_rdata` runs a tiny R script through `Rscript` that loads the `.Rdata` and writes
`mydata` (keeping `Id, Date`, plus a `tx_id` column such as `ORDER_NO` when requested) and
`covariates.dynamic` (keeping `Id, Cov.Date` plus any of `Gender, Income, high.season` the
file contains). Selecting columns inside R keeps the intermediate CSVs small.


In [2]:
# R script (thin extractor). Args: [1] .Rdata, [2] tx CSV, [3] cov CSV, [4] optional tx-id col.
# Dumps ONLY the columns the builder needs, so the intermediates stay small.
_R_EXTRACT = r"""
# Load bit64 if available: some files store Id as a bit64::integer64 column, which
# write.csv otherwise serialises as raw double bits (garbage). With bit64 loaded,
# as.character() recovers the correct decimal id so the tx<->cov join matches.
if (requireNamespace("bit64", quietly = TRUE)) suppressMessages(library(bit64))

args <- commandArgs(trailingOnly = TRUE)
infile <- args[1]; tx_out <- args[2]; cov_out <- args[3]
tx_id  <- if (length(args) >= 4 && nzchar(args[4])) args[4] else NA

env <- new.env()
load(infile, envir = env)                      # brings 'mydata' and 'covariates.dynamic' into env

tx  <- as.data.frame(get("mydata", envir = env))
cov <- as.data.frame(get("covariates.dynamic", envir = env))

# Coerce join/label keys to character so (possibly integer64) ids serialise correctly.
tx$Id  <- as.character(tx$Id)
cov$Id <- as.character(cov$Id)
if (!is.na(tx_id)) tx[[tx_id]] <- as.character(tx[[tx_id]])

# Transactions: who (Id) and when (Date), plus the order id when counting distinct orders.
tx_cols <- c("Id", "Date")
if (!is.na(tx_id)) tx_cols <- c(tx_cols, tx_id)
write.csv(tx[, tx_cols], tx_out, row.names = FALSE)

# Covariates: a customer x week calendar. Keep the keys plus any known covariate that
# exists in this file (intersect() silently drops the ones a given dataset lacks).
keep <- intersect(c("Id", "Cov.Date", "Gender", "Income", "high.season"), names(cov))
write.csv(cov[, keep], cov_out, row.names = FALSE)
"""


def extract_rdata(rdata_path: Path, tx_out: Path, cov_out: Path, tx_id: str | None = None) -> None:
    """Run the R extractor; raises with R's stderr if it fails."""
    result = subprocess.run(
        ["Rscript", "-e", _R_EXTRACT, str(rdata_path), str(tx_out), str(cov_out), tx_id or ""],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Rscript failed on {rdata_path.name}:\n{result.stderr}")


## Step 2 — Build a panel at a given frequency (pandas)

`build_panel` turns the extracted CSVs into a dense panel at the requested frequency. It
generalises the original `dataset_building.py` logic: customer set and date window come from
the data itself (no hardcoded dates), the time bucket is chosen by `freq`, the transaction
unit is chosen by `tx_id_col`, and which covariates to carry is a parameter.

The sub-year period column changes with `freq`:

| freq | sub-column | buckets per year |
|---|---|---|
| `week` | `week` = `dayofyear // 7` (clipped to 51) | 52 |
| `month` | `month` = calendar month | 12 |
| `year` | *(none)* | 1 |

Steps: bucket dates → count transactions per cell (rows, or distinct `tx_id_col`) → build the
full `Id × year × [sub]` grid → left-join counts (missing = 0) → optionally attach covariates
(`max` per cell, then forward/back-fill per customer).

In [3]:
# For each frequency: the sub-year column name and the full range of its buckets.
_PERIOD = {
    "week":  ("week",  range(0, 52)),   # weeks 0..51
    "month": ("month", range(1, 13)),   # months 1..12
    "year":  (None,    None),           # year only, no sub-column
}


def _add_period_cols(df: pd.DataFrame, date_col: str, freq: str) -> None:
    """Add `year` and (for week/month) the sub-year period column from a datetime column."""
    df["year"] = df[date_col].dt.year
    if freq == "week":
        # dayofyear // 7, clipped to 51 -> exactly 52 buckets per year (electronics convention)
        df["week"] = (df[date_col].dt.dayofyear // 7).clip(upper=51)
    elif freq == "month":
        df["month"] = df[date_col].dt.month


def period_key_cols(freq: str) -> list:
    """Grouping/join keys for a frequency, e.g. ['Id','year','week'] or ['Id','year']."""
    sub_col, _ = _PERIOD[freq]
    return ["Id", "year"] + ([sub_col] if sub_col else [])


def build_panel(tx_csv: Path, cov_csv: Path, freq: str = "week",
                covariate_cols=(), tx_id_col: str | None = None) -> pd.DataFrame:
    """Build a dense per-customer panel of transaction counts (+ optional covariates).

    Parameters
    ----------
    tx_csv : extracted transactions CSV (columns Id, Date [, tx_id_col]).
    cov_csv : extracted covariates CSV (columns Id, Cov.Date [, <covariates>]).
    freq : "week", "month", or "year".
    covariate_cols : covariate columns to carry (e.g. ("Gender","Income","high.season")).
    tx_id_col : if given, count DISTINCT values of this column per period (e.g. "ORDER_NO");
        otherwise count rows (one mydata row = one transaction).

    Returns columns: Id, year[, week|month], Transactions [, *covariate_cols].
    """
    sub_col, sub_range = _PERIOD[freq]
    keys = period_key_cols(freq)

    tx = pd.read_csv(tx_csv)
    cov = pd.read_csv(cov_csv)

    # Ids can be int in one table and string in the other; cast both to str so the join matches.
    tx["Id"] = tx["Id"].astype(str)
    cov["Id"] = cov["Id"].astype(str)
    tx["Date"] = pd.to_datetime(tx["Date"])
    cov["Cov.Date"] = pd.to_datetime(cov["Cov.Date"])

    # --- Transaction counts: one count per (customer, period) ---
    _add_period_cols(tx, "Date", freq)
    if tx_id_col:
        # Line-item data: a "transaction" is a distinct order, not a row.
        counts = tx.groupby(keys)[tx_id_col].nunique().reset_index(name="Transactions")
    else:
        counts = tx.groupby(keys).size().reset_index(name="Transactions")

    # --- Full rectangular grid: every customer x every year x every sub-period ---
    # Customer set and year span come from the covariate calendar (the cohort/window).
    ids = cov["Id"].unique()
    year_lo = int(cov["Cov.Date"].dt.year.min())
    year_hi = int(cov["Cov.Date"].dt.year.max())
    levels = [ids, range(year_lo, year_hi + 1)]
    if sub_col:
        levels.append(list(sub_range))
    grid = pd.MultiIndex.from_product(levels, names=keys).to_frame(index=False)

    # Left-join counts onto the grid; periods with no purchase become 0.
    panel = grid.merge(counts, on=keys, how="left")
    panel["Transactions"] = panel["Transactions"].fillna(0).astype(int)

    # --- Optional covariates ---
    if covariate_cols:
        _add_period_cols(cov, "Cov.Date", freq)
        # Collapse the weekly calendar to one row per period. high.season is a flag -> max;
        # Gender/Income are static per customer so max == the single value.
        cov_p = (cov.groupby(keys, as_index=False)
                    .agg({c: "max" for c in covariate_cols}))
        panel = panel.merge(cov_p, on=keys, how="left")
        for c in covariate_cols:
            panel[c] = panel.groupby("Id")[c].ffill().bfill()

    return panel.sort_values(keys).reset_index(drop=True)


## Step 3 — Build every dataset at every frequency

Each `.Rdata` is extracted once (the slow R step), then panels for all frequencies are built
from those temp CSVs and written to `Datasets/Dataset_clean/`. Built panels are kept in
`panels[(name, freq)]` for verification and validation.


In [4]:
panels = {}

for ds in DATASETS:
    with tempfile.TemporaryDirectory() as tmp:
        tx_csv = Path(tmp) / "tx.csv"
        cov_csv = Path(tmp) / "cov.csv"
        extract_rdata(DATASETS_DIR / ds["rdata"], tx_csv, cov_csv, ds["tx_id"])  # one R call per dataset
        for freq in FREQS:
            panel = build_panel(tx_csv, cov_csv, freq, ds["covariates"], ds["tx_id"])
            out = OUT_DIR / f"{ds['name']}_customer_{freq}_panel.csv"
            panel.to_csv(out, index=False)
            panels[(ds["name"], freq)] = panel
            print(f"wrote {out.name:<48} shape {str(panel.shape):>16}")


wrote apparel_customer_week_panel.csv                  shape     (1730872, 5)


wrote apparel_customer_month_panel.csv                 shape      (399432, 5)


wrote apparel_customer_year_panel.csv                  shape       (33286, 4)


wrote book_store_customer_week_panel.csv               shape      (506688, 4)


wrote book_store_customer_month_panel.csv              shape      (116928, 4)


...

wrote gift_retailer_customer_month_panel.csv           shape      (197952, 4)


wrote gift_retailer_customer_year_panel.csv            shape       (16496, 3)


wrote multichannel_customer_week_panel.csv             shape      (656136, 4)


wrote multichannel_customer_month_panel.csv            shape      (151416, 4)


wrote multichannel_customer_year_panel.csv             shape       (12618, 3)


## Step 4 — Verify the panels

Sanity checks for every built panel: no missing values, sub-period within range, integer
non-negative `Transactions`, and a complete grid (`rows == n_customers x n_years x buckets`).


In [5]:
def verify_panel(panel: pd.DataFrame, name: str, freq: str) -> None:
    sub_col, sub_range = _PERIOD[freq]
    buckets = len(sub_range) if sub_range else 1
    n_cust = panel["Id"].nunique()
    n_years = panel["year"].nunique()
    expected = n_cust * n_years * buckets

    assert panel.isna().sum().sum() == 0, f"{name}/{freq}: unexpected NaNs"
    assert (panel["Transactions"] >= 0).all(), f"{name}/{freq}: negative Transactions"
    assert panel["Transactions"].dtype.kind in "iu", f"{name}/{freq}: Transactions not integer"
    if sub_col:
        lo, hi = min(sub_range), max(sub_range)
        assert panel[sub_col].between(lo, hi).all(), f"{name}/{freq}: {sub_col} out of range"
    assert len(panel) == expected, f"{name}/{freq}: {len(panel)} rows != expected {expected}"

    print(f"{name:<19} {freq:<5} OK | rows={len(panel):>8} | customers={n_cust} | "
          f"years={panel['year'].min()}-{panel['year'].max()} | "
          f"tx max={panel['Transactions'].max():>3} | cols={list(panel.columns)}")


for (name, freq), panel in panels.items():
    verify_panel(panel, name, freq)


apparel             week  OK | rows= 1730872 | customers=3026 | years=1996-2006 | tx max=  3 | cols=['Id', 'year', 'week', 'Transactions', 'high.season']
apparel             month OK | rows=  399432 | customers=3026 | years=1996-2006 | tx max=  6 | cols=['Id', 'year', 'month', 'Transactions', 'high.season']
apparel             year  OK | rows=   33286 | customers=3026 | years=1996-2006 | tx max= 25 | cols=['Id', 'year', 'Transactions', 'high.season']
book_store          week  OK | rows=  506688 | customers=1218 | years=2007-2014 | tx max=  2 | cols=['Id', 'year', 'week', 'Transactions']


book_store          month OK | rows=  116928 | customers=1218 | years=2007-2014 | tx max=  6 | cols=['Id', 'year', 'month', 'Transactions']
book_store          year  OK | rows=    9744 | customers=1218 | years=2007-2014 | tx max= 23 | cols=['Id', 'year', 'Transactions']
electronicV2        week  OK | rows=  301756 | customers=829 | years=1998-2004 | tx max= 26 | cols=['Id', 'year', 'week', 'Transactions', 'Gender', 'Income', 'high.season']
electronicV2        month OK | rows=   69636 | customers=829 | years=1998-2004 | tx max= 27 | cols=['Id', 'year', 'month', 'Transactions', 'Gender', 'Income', 'high.season']
electronicV2        year  OK | rows=    5803 | customers=829 | years=1998-2004 | tx max= 66 | cols=['Id', 'year', 'Transactions', 'Gender', 'Income', 'high.season']
sporting_equipment  week  OK | rows=  453908 | customers=1247 | years=2014-2020 | tx max=  3 | cols=['Id', 'year', 'week', 'Transactions']
sporting_equipment  month OK | rows=  104748 | customers=1247 | years=2014-202

gift_retailer       month OK | rows=  197952 | customers=2062 | years=2001-2008 | tx max= 11 | cols=['Id', 'year', 'month', 'Transactions']
gift_retailer       year  OK | rows=   16496 | customers=2062 | years=2001-2008 | tx max= 50 | cols=['Id', 'year', 'Transactions']
multichannel        week  OK | rows=  656136 | customers=1402 | years=2004-2012 | tx max=  3 | cols=['Id', 'year', 'week', 'Transactions']


multichannel        month OK | rows=  151416 | customers=1402 | years=2004-2012 | tx max=  4 | cols=['Id', 'year', 'month', 'Transactions']
multichannel        year  OK | rows=   12618 | customers=1402 | years=2004-2012 | tx max= 10 | cols=['Id', 'year', 'Transactions']


## Step 5 — Validate Electronics V2 (weekly) against the original

Compare the freshly built **weekly** `electronicV2` panel against the known-good
`electronics_customer_week_panel.csv`. V2 has a wider window, so we outer-join on
`(Id, year, week)` and check that original-only keys = 0 and that all value columns match
exactly on the shared keys. A correct pipeline yields **0 mismatches**.


In [6]:
orig = pd.read_csv(OUT_DIR / "electronics_customer_week_panel.csv")
v2 = panels[("electronicV2", "week")].copy()

keys = ["Id", "year", "week"]
orig["Id"] = orig["Id"].astype(str)
v2["Id"] = v2["Id"].astype(str)

merged = orig.merge(v2, on=keys, how="outer", suffixes=("_orig", "_v2"), indicator=True)
counts = merged["_merge"].value_counts()
print("Key overlap (outer join on Id/year/week):")
print(f"  shared (both)     : {counts.get('both', 0)}")
print(f"  V2-only           : {counts.get('right_only', 0)}   (extra years -> expected)")
print(f"  original-only     : {counts.get('left_only', 0)}   (should be 0)")

both = merged[merged["_merge"] == "both"]
print("\nValue agreement on shared keys:")
all_ok = True
for col in ["Transactions", "Gender", "Income", "high.season"]:
    a, b = both[f"{col}_orig"], both[f"{col}_v2"]
    mism = ~((a == b) | (a.isna() & b.isna()))   # two NaNs count as equal
    n = int(mism.sum())
    all_ok &= (n == 0)
    print(f"  {col:<12}: {n} mismatch(es) / {len(both)}")
    if n:
        print(both.loc[mism, keys + [f"{col}_orig", f"{col}_v2"]].head())

print("\nRESULT:", "PASS - V2 reproduces the original on shared keys" if all_ok
      else "FAIL - see mismatches above")


Key overlap (outer join on Id/year/week):
  shared (both)     : 172432
  V2-only           : 129324   (extra years -> expected)
  original-only     : 0   (should be 0)

Value agreement on shared keys:
  Transactions: 0 mismatch(es) / 172432
  Gender      : 0 mismatch(es) / 172432
  Income      : 0 mismatch(es) / 172432
  high.season : 0 mismatch(es) / 172432

RESULT: PASS - V2 reproduces the original on shared keys
